# 양자화 인지 학습(QAT) 실습 — Fake Quantization과 STE

파이토치로 MNIST를 MLP로 학습한 뒤, **PTQ(사후 양자화)** 와 **QAT(양자화 인지 학습)** 를 비교합니다.

1. PTQ가 저비트에서 어떻게 무너지는지 (2비트에서 붕괴)
2. **Fake quantization** — forward에 양자화→복원을 끼우기
3. **STE(Straight-Through Estimator)** — round의 gradient가 0인 문제를 뚫기 (`autograd.Function`)
4. PTQ vs QAT 정확도 비교

> 참고: MIT 6.5940 *TinyML and Efficient Deep Learning Computing* (Song Han), Lecture 6. STE: Hinton 2012 · Bengio 2013.
> Colab에서는 위에서부터 순서대로 실행하면 됩니다 (런타임 → 모두 실행). CPU로도 몇 분이면 끝납니다.

## 0. 준비

In [ ]:
import copy
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. MNIST를 MLP로 학습

작은 3층 MLP(784 → 256 → 128 → 10)를 몇 에폭 학습합니다. 목적은 최고 성능이 아니라 **양자화해 볼 가중치**를 얻는 것입니다.

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
tr = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tf)
te = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tf)
trl = DataLoader(tr, batch_size=256, shuffle=True); tel = DataLoader(te, batch_size=512)

class MLP(nn.Module):
    def __init__(s):
        super().__init__(); s.fc1=nn.Linear(784,256); s.fc2=nn.Linear(256,128); s.fc3=nn.Linear(128,10)
    def forward(s, x):
        x=x.view(x.size(0),-1); x=F.relu(s.fc1(x)); x=F.relu(s.fc2(x)); return s.fc3(x)

def accuracy(m):
    m.eval(); c=t=0
    with torch.no_grad():
        for x,y in tel:
            x,y=x.to(device),y.to(device); c+=(m(x).argmax(1)==y).sum().item(); t+=y.size(0)
    return c/t

def train(m, ep, lr=1e-3):
    opt=torch.optim.Adam(m.parameters(), lr=lr)
    for _ in range(ep):
        m.train()
        for x,y in trl:
            x,y=x.to(device),y.to(device); opt.zero_grad(); F.cross_entropy(m(x),y).backward(); opt.step()

base = MLP().to(device); train(base, 3)
base_acc = accuracy(base)
print("FP32 baseline accuracy:", base_acc)

## 2. Fake quantization + STE

**Fake quantization**: 가중치를 정수 격자로 `round`한 뒤 다시 scale을 곱해 실수로 되돌립니다(값은 격자 위에 있지만 연산은 FP32).

**STE**: `round`는 거의 모든 점에서 미분이 0이라 그냥은 gradient가 흐르지 않습니다. `autograd.Function`으로 **forward=양자화 / backward=통과(항등함수)** 를 구현합니다. 표현 범위 밖은 clip 마스크로 0.

In [ ]:
class FakeQuantSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w, n_bits):
        qmax = 2 ** (n_bits - 1) - 1
        S = w.detach().abs().max() / qmax + 1e-12          # per-tensor symmetric scale
        q = torch.clamp(torch.round(w / S), -qmax, qmax)   # 격자로 스냅
        ctx.save_for_backward((w.abs() <= S * qmax).to(w.dtype))  # clip 마스크
        return q * S                                       # 복원 (fake-quant)
    @staticmethod
    def backward(ctx, g):
        (mask,) = ctx.saved_tensors
        return g * mask, None            # STE: 범위 안은 통과, 밖은 0

def fq(w, b): return FakeQuantSTE.apply(w, b)

## 3. PTQ — 사후 양자화 (저비트에서 붕괴)

학습된 가중치를 그대로 fake-quant 해서(재학습 없이) 정확도를 봅니다. 비트가 낮아질수록 무너집니다.

In [ ]:
def ptq(model, b):
    m = copy.deepcopy(model)
    with torch.no_grad():
        for layer in [m.fc1, m.fc2, m.fc3]:
            layer.weight.copy_(fq(layer.weight, b))
    return accuracy(m)

bits = [8, 4, 3, 2]
acc_ptq = {b: ptq(base, b) for b in bits}
for b in bits: print(f"{b}-bit PTQ: {acc_ptq[b]:.4f}")

## 4. QAT — fake-quant로 fine-tune

FP32 마스터 가중치를 유지한 채, forward에서 가중치를 fake-quant 한 상태로 몇 에폭 fine-tune 합니다. STE 덕분에 gradient가 마스터 가중치로 흘러, 모델이 "양자화된 상태에서 좋도록" 스스로를 조정합니다.

In [ ]:
class QMLP(nn.Module):
    def __init__(s, src, b):
        super().__init__(); s.b=b
        s.fc1=nn.Linear(784,256); s.fc2=nn.Linear(256,128); s.fc3=nn.Linear(128,10)
        s.load_state_dict(src.state_dict())     # 사전학습 FP 가중치로 초기화
    def forward(s, x):
        x=x.view(x.size(0),-1)
        x=F.relu(F.linear(x, fq(s.fc1.weight, s.b), s.fc1.bias))
        x=F.relu(F.linear(x, fq(s.fc2.weight, s.b), s.fc2.bias))
        return F.linear(x, fq(s.fc3.weight, s.b), s.fc3.bias)

def qat(src, b, ep=2, lr=3e-4):
    m = QMLP(src, b).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    for _ in range(ep):
        m.train()
        for x,y in trl:
            x,y=x.to(device),y.to(device); opt.zero_grad(); F.cross_entropy(m(x),y).backward(); opt.step()
    return accuracy(m)

acc_qat = {b: qat(base, b) for b in bits}
for b in bits: print(f"{b}-bit QAT: {acc_qat[b]:.4f}")

## 5. PTQ vs QAT 비교

같은 비트폭에서 두 방식의 정확도를 비교합니다. 2비트에서 PTQ는 랜덤 수준으로 붕괴하지만 QAT는 대부분 회복합니다.

In [ ]:
print(f"{'bits':>5} | {'PTQ':>8} | {'QAT':>8}")
for b in bits: print(f"{b:>5} | {acc_ptq[b]:>8.4f} | {acc_qat[b]:>8.4f}")
print(f"{'float':>5} | {base_acc:>8.4f} | {base_acc:>8.4f}")

xs = sorted(bits)
plt.figure(figsize=(6.8,4.3))
plt.axhline(base_acc, color="gray", ls=":", label=f"FP32 baseline ({base_acc:.3f})")
plt.plot(xs, [acc_ptq[b] for b in xs], "s--", color="#ef4444", label="PTQ (quantize after training)")
plt.plot(xs, [acc_qat[b] for b in xs], "o-",  color="#22c55e", label="QAT (fake-quant + STE)")
plt.title("QAT recovers accuracy where PTQ collapses (MNIST MLP)")
plt.xlabel("bits per weight"); plt.ylabel("test accuracy"); plt.xticks(xs); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## (보너스) STE 시각화 — forward 계단 / backward 기울기

`round`는 forward에서 계단 함수지만, STE는 backward에서 이를 항등함수(기울기 1)로 간주해 gradient를 통과시킵니다.

In [ ]:
S=0.34; qmax=1
x=np.linspace(-0.7,0.7,600); Qx=np.clip(np.round(x/S),-qmax,qmax)*S
fig,ax=plt.subplots(1,2,figsize=(11,3.7))
ax[0].plot(x,x,"--",color="#888",label="identity (y = w)")
ax[0].step(x,Qx,where="mid",color="#22c55e",lw=2,label="Q(w) = round(w/S)·S")
ax[0].set_title("forward: quantizer is a staircase"); ax[0].set_xlabel("w"); ax[0].set_ylabel("Q(w)"); ax[0].legend(fontsize=9); ax[0].grid(alpha=0.3)
inrange=(np.abs(x)<=S*qmax+S/2).astype(float)
ax[1].plot(x,np.zeros_like(x),color="#ef4444",lw=2,label="true dQ/dw = 0 (a.e.)")
ax[1].plot(x,inrange,color="#22c55e",lw=2,label="STE dQ/dw ~ 1 (in range)")
ax[1].set_ylim(-0.3,1.4); ax[1].set_title("backward: STE pretends it is the identity")
ax[1].set_xlabel("w"); ax[1].set_ylabel("gradient"); ax[1].legend(fontsize=9); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 마무리 & 더 해보기

- **PTQ**는 작은 모델·저비트에서 무너지고, **QAT**는 fake-quant + STE로 그 격차를 되돌립니다.
- **STE**의 핵심은 forward=양자화 / backward=항등함수. `autograd.Function` 하나로 구현됩니다.

**더 해볼 것**
1. activation도 fake-quant 해서 W/A 둘 다 양자화 (여기선 weight-only)
2. per-channel scale, learnable step size(LSQ)로 확장
3. BatchNorm folding과 함께 Conv 모델(LeNet 등)에 적용